# AI File Renamer (Ollama + Gemma)

Uses Gemma (via Ollama at `192.168.1.110:32125`) to check whether each file name in a target directory meets your naming expectations, and renames files that don't.

**Workflow:**
1. Set configuration (host, model, directory, dry-run flag)
2. Edit the naming context (your expectations/rules)
3. Test the connection
4. Run the main loop — review output, then set `DRY_RUN = False` to apply

> **Safety:** `DRY_RUN = True` by default. No files are modified until you flip it. Existing files are never overwritten (collisions are skipped).

In [63]:
# Install dependencies if needed (run once):
# %pip install requests pandas

import json
import re
from pathlib import Path

import requests

In [64]:
# ============================ CONFIGURATION ============================

OLLAMA_HOST = "http://192.168.1.110:32125"
MODEL       = "gemma4:e4b"

TARGET_DIR  = r"Z:\Misc"   # <-- EDIT: directory to scan
FILE_GLOB   = "*.CBZ"                        # e.g. "*.pdf", "*.jpg", "*" for all files

BATCH_SIZE  = 1           # max files evaluated per LLM call
MAX_FILES   = 1000          # maximum files processed in this run

DRY_RUN     = False                       # True = preview only; False = actually rename

In [65]:
# ===================== NAMING EXPECTATIONS (CONTEXT) =====================
# This is your "expanded context". Describe your naming conventions here —
# Gemma will use this to judge each file name and propose corrections.
# Be as detailed as you like: patterns, examples, do's and don'ts.

NAMING_CONTEXT = """
You are a filename normalization assistant for comic book archives (.cbz).

Your task is to rename the input filename according to the rules below. Return ONLY the proposed new filename and a short explanation. Do not invent information that is not present or clearly implied by the original filename.

IMPORTANT: Your job is to REMOVE, REORDER, and NORMALIZE information that already exists in the filename. Do not translate Japanese yourself. However, if an English translation is already present in the filename alongside a romanized Japanese title, prefer the existing English translation.

==================================================
RENAME RULES — APPLY IN THIS ORDER
==================================================

1. PRESERVE THE FILE EXTENSION
- Always keep the original file extension unchanged, including capitalization.
- Examples: .cbz remains .cbz, .CBZ remains .CBZ.

2. IDENTIFY THE AUTHOR / CIRCLE
- The preferred format is:
  [Author]
  or:
  [Author (Circle)]
- If the original filename contains an author and circle in this form, preserve them.
- Do not invent an author or circle.
- The author/circle bracket is the ONLY bracketed information that should normally remain in the final filename.
- Examples:
  [Akira Toriyama]
  [Ayashii Bochi (PINTA)]

3. REMOVE RELEASE, SCANLATION, LANGUAGE, AND ARCHIVE TAGS
Remove tags and metadata such as:
- [English]
- [EN]
- [Japanese]
- [Digital]
- [Completed]
- [Scan]
- [Scans]
- [Scanlation]
- [Decensored]
- [Censored]
- [Uncensored]
- [Translated]
- [Translation]
- [Saint Quartz Scans]
- and similar release/scanlation/language/group tags.

The author/circle bracket is the exception and should be preserved.

4. REMOVE INTERNAL RELEASE / ARCHIVE IDENTIFIERS
Remove meaningless internal identifiers such as:
- (AC2)
- (AC1)
- release numbers
- uploader identifiers
- archive identifiers
- similar metadata

These are not part of the work's title.

5. DISTINGUISH THE ACTUAL TITLE FROM FRANCHISE / SERIES METADATA
A parenthesized phrase is NOT automatically part of the title.

If a phrase identifies the franchise, game, anime, manga, or larger series that the work is based on, remove it when it is clearly metadata rather than part of the actual work title.

Example:
Simply Living with Ushiwakamaru (Fate Grand Order)
becomes:
Simply Living with Ushiwakamaru

However, preserve parentheses when they are clearly part of the actual published title.

6. ROMANIZED JAPANESE TITLE + ENGLISH TITLE
This is a CRITICAL RULE.

If the filename contains both:
- a Japanese title written using Latin/romanized characters
AND
- an English translation of that title

KEEP ONLY THE ENGLISH TITLE.

The romanized Japanese title must be removed.

The English translation may appear immediately after the romanized Japanese title without parentheses or another delimiter.

Examples:

Ushiwakamaru to Kurasu dake Simply Living with Ushiwakamaru
-> Simply Living with Ushiwakamaru

One Piece (Wan Pisu)
-> One Piece

Kimi no Na wa Your Name
-> Your Name

Boku no Hero Academia My Hero Academia
-> My Hero Academia

DO NOT:
- concatenate both versions
- keep both versions
- assume the romanized Japanese version is part of the English title
- require parentheses to recognize the English translation

When two consecutive title phrases appear to represent the same work, and one is romanized Japanese while the other is an existing English translation, prefer the English translation.

IMPORTANT:
Do NOT translate Japanese text yourself.
Only use an English title if an English title is already present in the original filename.

7. DO NOT INVENT INFORMATION
Never invent:
- authors
- circles
- titles
- translations
- chapter numbers
- volume numbers
- publication information

Only use information present in or clearly implied by the original filename.

8. PRESERVE CHAPTER / VOLUME INFORMATION
Chapter and volume information should be retained and placed at the END of the title.

Examples:
v01
v02
v01-v02
c001
c001-c005
Ch. 05
Ch. 05-06
Chapter 12

Examples:
[Author] My Title v01.cbz
[Author] My Title c001-c005.cbz

Do not remove legitimate chapter or volume information.

9. TITLE CASE
Use Title Case for the final title:
- Capitalize major words.
- Do not use ALL CAPS unless it is clearly a proper name/acronym.
- Do not change the spelling of names.

10. NORMALIZE SPACING
- Use single spaces between words.
- Remove double spaces.
- Remove leading/trailing spaces.
- Remove unnecessary hyphens, underscores, and dots used as separators.
- Remove trailing empty parentheses.
- Remove emojis and other obvious filename junk.

11. DO NOT USE UNNECESSARY BRACKETS
The final filename should normally contain:
[Author (Circle)] Title Chapter.ext

The author/circle bracket is allowed.

Do not retain:
[English]
[Digital]
[Scan]
[Decensored]
[Series Name]
or other metadata brackets.

12. FINAL FORMAT

If author/circle is known:

[Author (Circle)] Title.ext

If only author is known:

[Author] Title.ext

If author/circle is unknown:

Title.ext

If chapter or volume information exists:

[Author (Circle)] Title c001.ext
[Author (Circle)] Title v01.ext
[Author (Circle)] Title c001-c005.ext

==================================================
CRITICAL EXAMPLE
==================================================

INPUT:

(AC2) [Ayashii Bochi (PINTA)] Ushiwakamaru to Kurasu dake Simply Living with Ushiwakamaru (Fate Grand Order) [English] [Saint Quartz Scans] [Decensored].cbz

OUTPUT:

[Ayashii Bochi (PINTA)] Simply Living with Ushiwakamaru.cbz

WHY:

(AC2)
= internal archive/release metadata
= REMOVE

[Ayashii Bochi (PINTA)]
= author and circle
= KEEP

Ushiwakamaru to Kurasu dake
= romanized Japanese title
= REMOVE because an English translation is already present

Simply Living with Ushiwakamaru
= existing English title
= KEEP

(Fate Grand Order)
= franchise metadata
= REMOVE

[English]
= language tag
= REMOVE

[Saint Quartz Scans]
= scanlation group
= REMOVE

[Decensored]
= release/censorship tag
= REMOVE

Final filename:
[Ayashii Bochi (PINTA)] Simply Living with Ushiwakamaru.cbz

==================================================
ADDITIONAL EXAMPLES
==================================================

GOOD:
[Akira Toriyama] Dragon Ball v01.cbz

GOOD:
[Eiichiro Oda] One Piece c001-c015.cbz

GOOD:
Solo Leveling Ch. 05.cbz

GOOD:
[Author (Circle)] Simply Living with Ushiwakamaru.cbz

BAD:
[English] One Piece v01 [uncensored].cbz

Correct:
One Piece v01.cbz

BAD:
dragon_ball_v01_[Akira_Toriyama].cbz

Correct:
[Akira Toriyama] Dragon Ball v01.cbz

BAD:
ONE PIECE c001 [Digital] [EN].cbz

Correct:
One Piece c001.cbz

BAD:
[Author] Ushiwakamaru to Kurasu dake Simply Living with Ushiwakamaru.cbz

Correct:
[Author] Simply Living with Ushiwakamaru.cbz

BAD:
[Author] Simply Living with Ushiwakamaru (Fate Grand Order) [English] [Saint Quartz Scans].cbz

Correct:
[Author] Simply Living with Ushiwakamaru.cbz

==================================================
FINAL CHECK BEFORE RETURNING A NAME
==================================================

Before producing the result, verify:

- Is the original extension preserved exactly?
- Is the author/circle preserved if present?
- Were scanlation/language/release tags removed?
- Were internal archive identifiers removed?
- Was unnecessary franchise metadata removed?
- If both romanized Japanese and English title text existed, was ONLY the English title retained?
- Were chapter/volume numbers preserved?
- Did I avoid inventing anything?
- Are there single spaces between words?
- Is the result in the required format?
- Is the final filename clean and concise?

Return the normalized filename as the rename target.
"""

In [66]:
# ================ CONNECTION TEST ================
# Verifies Ollama is reachable and shows available models.

r = requests.get(f"{OLLAMA_HOST}/api/tags", timeout=10)
r.raise_for_status()

models = [m["name"] for m in r.json().get("models", [])]
print(f"Connected to {OLLAMA_HOST}")
print("Available models:")
for name in models:
    marker = "  <-- selected" if name == MODEL else ""
    print(f"  - {name}{marker}")

if MODEL not in models:
    print(f"\nWARNING: '{MODEL}' not found on the server. Check the MODEL setting.")

Connected to http://192.168.1.110:32125
Available models:
  - gemma4:e4b  <-- selected


In [67]:
# ===================== CORE FUNCTIONS =====================

def check_filenames_batch(filenames: list[str]) -> list[dict]:
    """Ask Gemma to evaluate a batch of file names in ONE request.

    Returns a list of verdict dicts in the same order as `filenames`.
    """
    numbered = "\n".join(f"{i}. {name}" for i, name in enumerate(filenames, start=1))
    prompt = f"""You are a strict file-naming assistant.

Naming expectations:
{NAMING_CONTEXT.strip()}

Evaluate each of the following {len(filenames)} file names (numbered list):
{numbered}

Instructions:
1. For each name, decide whether it meets the expectations above.
2. If not, suggest a corrected name that does. Derive it from the original name — never invent information (authors, titles, volumes) that is not present or implied in the original name.
3. Always keep the original file extension unchanged (including its case).
4. If a name already meets expectations, set suggested_name to an empty string.
5. Results MUST appear in the SAME ORDER as the input list.

Respond with ONLY a JSON object and no other text, in exactly this shape:
{{"results": [{{"filename": "exact original name", "meets_expectations": true or false, "suggested_name": "corrected name or empty string", "reason": "one short sentence"}}, ...]}}"""

    response = requests.post(
        f"{OLLAMA_HOST}/api/generate",
        json={
            "model": MODEL,
            "prompt": prompt,
            "stream": False,
            "format": "json",              # force JSON output
            "options": {"temperature": 0.1},
        },
        timeout=600,
    )
    response.raise_for_status()
    raw = response.json().get("response", "")
    data = parse_json_response(raw)
    results = data.get("results") if isinstance(data, dict) else data
    if not isinstance(results, list):
        raise ValueError(f"Unexpected response shape: {raw!r}")
    return align_results(filenames, results)


def align_results(filenames: list[str], results: list[dict]) -> list[dict]:
    """Align model results back to the input order; fill gaps with REVIEW entries."""
    by_name = {}
    for r_ in results:
        if isinstance(r_, dict) and r_.get("filename"):
            by_name[str(r_["filename"])] = r_
    aligned = []
    for name in filenames:
        r_ = by_name.get(name)
        if r_ is None:  # model omitted this file
            r_ = {"filename": name, "meets_expectations": None,
                  "suggested_name": "", "reason": "no verdict returned by model"}
        aligned.append(r_)
    return aligned


def parse_json_response(raw):
    """Parse the model's JSON, tolerating stray text around it."""
    raw = raw.strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", raw, re.DOTALL)
        if match:
            return json.loads(match.group(0))
        raise ValueError(f"Model did not return JSON: {raw!r}")


def sanitize_name(name: str, original_suffix: str) -> str:
    """Make the suggested name safe and ensure the extension is preserved."""
    name = name.replace("\\", "_").replace("/", "_").strip().strip(".")
    if original_suffix and not name.lower().endswith(original_suffix.lower()):
        stem = Path(name).stem if Path(name).suffix else name
        name = stem + original_suffix
    return name


def safe_rename(path: Path, new_name: str, dry_run: bool = True) -> str:
    """Rename a file, with collision protection and dry-run support."""
    new_path = path.with_name(new_name)
    if new_path == path:
        return "unchanged"
    if new_path.exists():
        return f"SKIPPED (target already exists: {new_name})"
    if dry_run:
        return f"DRY RUN -> would rename to: {new_name}"
    path.rename(new_path)
    return f"renamed to: {new_name}"

In [ ]:
# ============================ MAIN LOOP ============================

target = Path(TARGET_DIR)
assert target.is_dir(), f"Directory not found: {target}"

files = [p for p in sorted(target.glob(FILE_GLOB)) if p.is_file()][:MAX_FILES]
batches = [files[i:i + BATCH_SIZE] for i in range(0, len(files), BATCH_SIZE)]

print(f"Mode: {'DRY RUN (no changes will be made)' if DRY_RUN else 'LIVE (files WILL be renamed)'}")
print(f"Found {len(files)} file(s) in {target} -> {len(batches)} batch(es) "
      f"of up to {BATCH_SIZE} (limit: {MAX_FILES})\n{'=' * 60}\n")

results = []

for batch_num, batch in enumerate(batches, start=1):
    print(f"--- Batch {batch_num}/{len(batches)} ({len(batch)} files) ---\n")

    try:
        verdicts = check_filenames_batch([p.name for p in batch])
    except Exception as e:
        print(f"[ERROR] batch {batch_num} failed: {e}\n")
        for path in batch:
            results.append({"old_name": path.name, "new_name": path.name,
                            "status": "ERROR", "reason": str(e), "action": "none"})
        continue

    for path, verdict in zip(batch, verdicts):
        old_name = path.name

        meets     = verdict.get("meets_expectations")
        suggested = str(verdict.get("suggested_name") or "").strip()
        reason    = str(verdict.get("reason") or "")

        if meets is True:
            status, new_name, action = "OK", old_name, "kept"
        elif not suggested or suggested == old_name:
            status, new_name, action = "REVIEW", old_name, "no suggestion returned"
        else:
            new_name = sanitize_name(suggested, path.suffix)
            status   = "RENAME"
            action   = safe_rename(path, new_name, dry_run=DRY_RUN)

        results.append({"old_name": old_name, "new_name": new_name,
                        "status": status, "reason": reason, "action": action})
        print(f"[{status}] {old_name}")
        if status != "OK":
            print(f"   -> {new_name}")
        print(f"   {reason} | {action}\n")

print("=" * 60)
counts = {}
for r_ in results:
    counts[r_["status"]] = counts.get(r_["status"], 0) + 1
print("Summary:", ", ".join(f"{k}: {v}" for k, v in sorted(counts.items())))

Mode: LIVE (files WILL be renamed)
Found 445 file(s) in Z:\Misc -> 445 batch(es) of up to 1 (limit: 1000)

--- Batch 1/445 (1 files) ---

[REVIEW] (C84) [Yokoshimanchi. (Ash Yokoshima)] 3 ANGELS SHORT Full Blossom  0.8 cafe au lait [English] [Tigoris Translates].cbz
   -> (C84) [Yokoshimanchi. (Ash Yokoshima)] 3 ANGELS SHORT Full Blossom  0.8 cafe au lait [English] [Tigoris Translates].cbz
   no verdict returned by model | no suggestion returned

--- Batch 2/445 (1 files) ---

[RENAME] (Sensei no Archive 6)  Byoujaku Bishoujo Hacker A Certain Day's Super Genius Lewd Delicate Beauty Hacker (.cbz
   -> Byoujaku Bishoujo Hacker A Certain Day's Super Genius Lewd Delicate Beauty Hacker.cbz
   The initial parenthetical phrase appears to be a volume/archive identifier and should be removed, leaving the main title. | renamed to: Byoujaku Bishoujo Hacker A Certain Day's Super Genius Lewd Delicate Beauty Hacker.cbz

--- Batch 3/445 (1 files) ---

[RENAME] 1 (1).cbz
   -> 1.cbz
   Removed unneces

In [ ]:
# ===================== RESULTS TABLE (optional) =====================
# Review the full results as a table. Useful before a live run.

import pandas as pd

df = pd.DataFrame(results)
df

,old_name,new_name,status,reason,action
0,(C101) [Jingai Makyou (Inue Shinsuke)] Imouto ...,[Jingai Makyou (Inue Shinsuke)] My Little Sist...,RENAME,"Removed internal identifiers, language tags, a...",renamed to: [Jingai Makyou (Inue Shinsuke)] My...
1,(C84) [enuma elish (Yukimi)] Valhallagatari (B...,[enuma elish (Yukimi)] Valhallagatari.cbz,RENAME,"Removed internal identifiers, language tags, f...",renamed to: [enuma elish (Yukimi)] Valhallagat...
2,(C84) [Yokoshimanchi. (Ash Yokoshima)] 3 ANGEL...,(C84) [Yokoshimanchi. (Ash Yokoshima)] 3 ANGEL...,REVIEW,no verdict returned by model,no suggestion returned
3,(C90) [Melty Pot (Mel)] Happy Style! 5 (Yuyush...,[Melty Pot (Mel)] Happy Style! 5.cbz,RENAME,"Removed internal identifiers, language tags, a...",renamed to: [Melty Pot (Mel)] Happy Style! 5.cbz
4,(C94) [Hiiro no Kenkyuushitsu (Hitoi)] NeuTRal...,[Hiiro no Kenkyuushitsu (Hitoi)] NeuTRal Actor...,RENAME,"Removed internal identifiers, language tags, a...",renamed to: [Hiiro no Kenkyuushitsu (Hitoi)] N...
...,...,...,...,...,...
440,VictimGirls 19 JEZEBEL AMAZONES.cbz,VictimGirls 19 JEZEBEL AMAZONES.cbz,REVIEW,The name is already concise and contains no re...,no suggestion returned
441,vvvvv.cbz,vvvvv.cbz,REVIEW,The filename is already clean and contains no ...,no suggestion returned
442,Wakuwaku One7.cbz,Wakuwaku One7.cbz,REVIEW,The name is already clean and follows the gene...,no suggestion returned
443,ZZ_09BOR~N.cbz,ZZ_09BOR~N.cbz,REVIEW,The filename appears to be a unique identifier...,no suggestion returned


## Notes

- **Dry run**: keep `DRY_RUN = True` while tuning your naming context. Review the proposals, then set it to `False` and re-run the main loop.
- **Expanded context**: the more specific `NAMING_CONTEXT` is (patterns, good/bad examples), the better Gemma's suggestions will be. Few-shot examples help small models a lot.
- **Statuses**: `OK` (name is fine), `RENAME` (suggestion applied or previewed), `REVIEW` (model flagged it but gave no usable suggestion), `ERROR` (request/parsing failed).
- **Safety**: extensions are always preserved, existing files are never overwritten, and renames stay within the same directory.
- **Reruns**: already-renamed files will come back `OK` on the next pass, so the notebook is safe to re-run.